# KoHRM-Text-1.4B Colab T4 Inference and Domain Probes

This notebook loads the latest public KoHRM-Text-1.4B checkpoint without `transformers`, so it avoids the Colab `torchvision::nms` / custom `HrmTextConfig` import failure. It is a smoke and behavior probe for the current public checkpoint, not a final benchmark harness.

KoHRM uses the HRM-Text training format:

```text
<|im_start|><condition_token>instruction<|im_end|>response<|box_end|>
```

Use `direct` / `<|object_ref_start|>` for answer-only, JSON-only, command-only, and code-only outputs. Korean law/wiki/finance probes below use Korean prompts; terminal, tool-call, and coding probes use English prompts because that better matches the current training mix for those tasks.

## 1. Install Dependencies

The runtime intentionally does not import `transformers`. `tokenizers` is pinned below `0.23.1` to avoid conflicts with Colab images that already contain `transformers 5.x`.

In [ ]:
!pip -q install -U huggingface_hub hf_transfer safetensors
!pip -q install --force-reinstall -q "tokenizers>=0.22.0,<0.23.1"

## 2. Runtime Settings

`MAX_SEQ_LEN=512` is the safest T4 default. Raise it to `768` only if the first load leaves enough free VRAM.

In [ ]:
import os
import json
import gc
import importlib.util
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

REPO_ID = "LLM-OS-Models/KoHRM-Text-1.4B"
REVISION = "main"
LOCAL_DIR = Path("/content/KoHRM-Text-1.4B")
HELPER_PATH = LOCAL_DIR / "kohrm_colab_generate.py"
MAX_SEQ_LEN = 512

STRICT_SETTINGS = {
    "max_seq_len": MAX_SEQ_LEN,
    "temperature": 0.0,
    "top_p": 1.0,
    "repetition_penalty": 1.20,
    "no_repeat_ngram_size": 4,
    "condition_token": "<|object_ref_start|>",
}

print("repo:", REPO_ID)
print("revision:", REVISION)
print("local_dir:", LOCAL_DIR)
print("max_seq_len:", MAX_SEQ_LEN)

## 3. Download Latest Public Checkpoint

The public model repo is expected to contain `config.json`, `tokenizer.json`, `model.safetensors`, and the model card. If the lightweight helper is not present in the model repo, the notebook clones the GitHub repo and copies the helper from `notebooks/`.

In [ ]:
from huggingface_hub import snapshot_download

LOCAL_DIR.mkdir(parents=True, exist_ok=True)
patterns = [
    "README.md",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "model.safetensors",
    "kohrm_colab_generate.py",
    "notebooks/kohrm_colab_generate.py",
]

snapshot_download(
    repo_id=REPO_ID,
    repo_type="model",
    revision=REVISION,
    local_dir=str(LOCAL_DIR),
    local_dir_use_symlinks=False,
    allow_patterns=patterns,
)

nested_helper = LOCAL_DIR / "notebooks" / "kohrm_colab_generate.py"
if not HELPER_PATH.exists() and nested_helper.exists():
    HELPER_PATH.write_text(nested_helper.read_text(encoding="utf-8"), encoding="utf-8")

if not HELPER_PATH.exists():
    repo = Path("/content/KoHRM-text")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/LLM-OS-Models/KoHRM-text", str(repo)],
            check=True,
        )
    HELPER_PATH.write_text((repo / "notebooks" / "kohrm_colab_generate.py").read_text(encoding="utf-8"), encoding="utf-8")

for name in ["config.json", "tokenizer.json", "model.safetensors", "README.md", "kohrm_colab_generate.py"]:
    path = LOCAL_DIR / name
    print(f"{name}: {path.exists()} {path.stat().st_size / 2**20:.2f} MiB" if path.exists() else f"{name}: missing")

## 4. Inspect Config and Prompt Format

Do not use `AutoTokenizer` or `AutoModelForCausalLM` here. The current public export uses a custom HRM-Text architecture, and the helper below loads it directly from `safetensors`.

In [ ]:
spec = importlib.util.spec_from_file_location("kohrm_colab_generate", HELPER_PATH)
kohrm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(kohrm)

config = json.loads((LOCAL_DIR / "config.json").read_text(encoding="utf-8"))
print(json.dumps({
    "model_type": config.get("model_type"),
    "architectures": config.get("architectures"),
    "vocab_size": config.get("vocab_size"),
    "hidden_size": config.get("hidden_size"),
    "num_hidden_layers": config.get("num_hidden_layers"),
    "num_attention_heads": config.get("num_attention_heads"),
    "H_cycles": config.get("H_cycles"),
    "L_cycles": config.get("L_cycles"),
    "max_position_embeddings": config.get("max_position_embeddings"),
    "prefix_lm": config.get("prefix_lm"),
}, indent=2, ensure_ascii=False))

from tokenizers import Tokenizer
raw_tok = Tokenizer.from_file(str(LOCAL_DIR / "tokenizer.json"))
for token in ["<|im_start|>", "<|object_ref_start|>", "<|object_ref_end|>", "<|quad_start|>", "<|quad_end|>", "<|im_end|>", "<|box_end|>"]:
    print(f"{token:22s}", raw_tok.token_to_id(token))

example = kohrm.format_kohrm_prompt("Return one bash command only.")
print("wrapped prompt:", example)

## 5. Load Model Once

On a T4, loading can take a few minutes. The helper uses PyTorch scaled-dot-product attention and a static KV cache. It is slower than the training-time FlashAttention path, but it is portable enough for Colab smoke tests.

In [ ]:
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(torch.cuda.get_device_name(0))

model, tokenizer, cfg = kohrm.load_kohrm(LOCAL_DIR, max_gpu_memory_gib=14.0)
print("loaded dtype:", next(model.parameters()).dtype)
print("loaded device:", next(model.parameters()).device)
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory free/total GiB after load: {free / 2**30:.2f}/{total / 2**30:.2f}")

## 6. Training-Aligned Probe Cases

Use deterministic decoding first. For terminal/coding/tool-call tests, the prompt language is English because the current prepared data has stronger English command/code distribution. Korean prompts are used for Korean legal, wiki-style, and finance-domain probes.

In [ ]:
TEST_CASES = [
    {
        "name": "ko_legal_json_direct",
        "lang": "ko",
        "expect": "json",
        "max_new_tokens": 128,
        "prompt": """다음 한국 법령/행정규칙 발췌문에서 조문명, 적용 대상, 핵심 의무를 JSON 객체 하나로만 추출하세요. JSON 밖의 설명은 쓰지 마세요.

[문서명]
진안군 홍보대사 운영 조례

[조문]
제5조 (보상)
① 홍보대사는 무보수 명예직으로 한다.
② 군수는 홍보대사가 임무 수행을 위하여 활동하는 경우 예산의 범위 안에서 홍보활동에 직접 소요되는 실 경비로 숙식비, 차량운행 경비, 기타 비용과 격려금품을 지급할 수 있다.""",
    },
    {
        "name": "ko_wiki_grounded_summary",
        "lang": "ko",
        "expect": "korean_text",
        "max_new_tokens": 96,
        "prompt": """다음 글만 근거로 핵심 내용을 한국어로 3문장 이내로 요약하세요. 글에 없는 사실은 추가하지 마세요.

[글]
훈민정음은 조선 세종이 창제한 문자 체계이다. 창제 목적은 백성이 자신의 뜻을 쉽게 글로 표현하도록 돕는 데 있었다. 자음은 발음 기관의 모양을 본떠 만들었고, 모음은 하늘, 땅, 사람의 원리를 바탕으로 구성되었다.""",
    },
    {
        "name": "ko_finance_short",
        "lang": "ko",
        "expect": "korean_text",
        "max_new_tokens": 96,
        "prompt": "환율 변동이 개인 투자에 미치는 영향과 대비 전략을 한국어로 4문장 이내로 설명하세요. 같은 표현을 반복하지 마세요.",
    },
    {
        "name": "en_terminal_command_only",
        "lang": "en",
        "expect": "one_line_command",
        "max_new_tokens": 64,
        "prompt": "Return one bash command only. No explanation. Task: find the 10 largest files under the current directory, excluding .git, sorted by size descending.",
    },
    {
        "name": "en_tool_call_json",
        "lang": "en",
        "expect": "json",
        "max_new_tokens": 96,
        "prompt": "Return one JSON object only for a terminal tool call. Schema: {\"tool\": \"shell\", \"args\": {\"cmd\": string}}. Task: print current disk usage for the current directory in human-readable form.",
    },
    {
        "name": "en_python_code_only",
        "lang": "en",
        "expect": "code",
        "max_new_tokens": 128,
        "prompt": "Write Python code only. Define a function top_k_lengths(items, k) that returns the k longest strings from items, preserving original order for ties.",
    },
]


def validate_output(expect, text):
    text = text.strip()
    if expect == "json":
        try:
            json.loads(text)
            return "ok: valid JSON"
        except Exception as exc:
            return f"warn: invalid JSON ({exc.__class__.__name__})"
    if expect == "one_line_command":
        if "\n" in text:
            return "warn: more than one line"
        if any(marker in text for marker in ["```", "We need", "Step", "1."]):
            return "warn: contains explanation/formatting"
        return "ok: one line"
    if expect == "code":
        return "ok: contains def" if "def " in text else "warn: no function definition detected"
    if expect == "korean_text":
        return "warn: empty" if not text else "ok: non-empty text"
    return "unchecked"

results = []
for case in TEST_CASES:
    print("=" * 80)
    print("case:", case["name"])
    print("expect:", case["expect"])
    print("prompt:", case["prompt"])
    output = kohrm.generate_from_loaded(
        model,
        tokenizer,
        cfg,
        case["prompt"],
        max_new_tokens=case["max_new_tokens"],
        **STRICT_SETTINGS,
    )
    verdict = validate_output(case["expect"], output)
    results.append({"case": case["name"], "expect": case["expect"], "verdict": verdict, "output": output})
    print("verdict:", verdict)
    print("--- output ---")
    print(output)

print("=" * 80)
print(json.dumps([{k: r[k] for k in ["case", "expect", "verdict"]} for r in results], indent=2, ensure_ascii=False))

## 7. Optional Low-Temperature Retry

Use this only after deterministic decoding. If deterministic output repeats badly, a small non-zero temperature plus repetition penalty can sometimes reveal whether the issue is decoding brittleness or checkpoint behavior.

In [ ]:
RETRY_CASE_NAMES = {"ko_finance_short", "en_terminal_command_only"}
RETRY_SETTINGS = dict(STRICT_SETTINGS)
RETRY_SETTINGS.update({
    "temperature": 0.2,
    "top_p": 0.85,
    "repetition_penalty": 1.22,
    "no_repeat_ngram_size": 4,
})

for case in TEST_CASES:
    if case["name"] not in RETRY_CASE_NAMES:
        continue
    print("=" * 80)
    print("retry case:", case["name"])
    output = kohrm.generate_from_loaded(
        model,
        tokenizer,
        cfg,
        case["prompt"],
        max_new_tokens=case["max_new_tokens"],
        **RETRY_SETTINGS,
    )
    print(output)

## 8. Interpretation Checklist

A bad terminal-command output in Korean does not necessarily prove the model cannot learn terminal tasks; the current training mix is stronger for English terminal/coding prompts. A repeated phrase in Korean finance or hallucinated legal rewrite is a checkpoint-quality signal and usually means the public checkpoint still needs more pretraining and/or a small behavior LoRA/SFT pass.

For a final model-quality decision, compare checkpoints with the same prompts, same condition token, same deterministic settings, and the same helper version. The probe is intentionally strict: JSON-only, command-only, and code-only tasks should be judged by validators, not by whether the text looks fluent.